Author: Nils Wikström @ university west niwi0007@student.hv.se


I really wanted to use tensorflow but it dose not want to work with CUDA on widows but torch seems to work just fine for some reason so ill be using that instead hope that is fine

Imports 

In [1]:
import torch
import torch.nn as nn
import tifffile
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

Check if Cuda and GPU is available

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version (PyTorch):", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
CUDA version (PyTorch): 12.4
GPU count: 1
GPU 0: NVIDIA GeForce RTX 4070


Configuration

In [ ]:
# Config
TIF_PATH = './Raw/CAPELLA_C13_SP_SLC_HH_20251112022441_20251112022453.tif'
SCALE_FACTOR = 4
PATCH_SIZE_HR = 128
PATCH_SIZE_LR = PATCH_SIZE_HR // SCALE_FACTOR
PATCH_STRIDE = 64
BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4
TRAIN_SPLIT = 0.85
LOOKS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")
print(f"Scale factor: {SCALE_FACTOR}x")
print(f"HR patch size: {PATCH_SIZE_HR}x{PATCH_SIZE_HR}")
print(f"LR patch size: {PATCH_SIZE_LR}x{PATCH_SIZE_LR}")
print(f"Multilook: {LOOKS} looks")

Using device: cuda
Scale factor: 4x
HR patch size: 128x128
LR patch size: 32x32
Multilook: 4 looks


In [ ]:
LOOKS = 4  # multilook factor

#  multilook image dimensions without loading the full image
with tifffile.TiffFile(TIF_PATH) as tif:
    H_slc, W_slc = tif.pages[0].shape[:2]

H_ml = H_slc // LOOKS
W_ml = W_slc // LOOKS
print(f"Original SLC size: {H_slc} x {W_slc}")
print(f"After {LOOKS}-look size: {H_ml} x {W_ml}")

# Generate coords based on multilook dimensions
coords = [
    (y, x)
    for y in range(0, H_ml - PATCH_SIZE_HR + 1, PATCH_STRIDE)
    for x in range(0, W_ml - PATCH_SIZE_HR + 1, PATCH_STRIDE)
]
print(f"Total patches: {len(coords)}")

Original SLC size: 12197 x 38982
After 4-look size: 3049 x 9745
Total patches: 6946


In [ ]:

print("Loading and multilooing for variance analysis...")
with tifffile.TiffFile(TIF_PATH) as tif:
    raw_data = np.abs(tif.pages[0].asarray()).astype(np.float32)

raw_data = multilook(raw_data, looks=LOOKS)

print("Computing patch variances")
patch_variances = []
for y, x in coords:
    block = raw_data[y:y+PATCH_SIZE_HR, x:x+PATCH_SIZE_HR]
    patch_variances.append(block.var())

patch_variances = np.array(patch_variances)
threshold = np.percentile(patch_variances, 60)
filtered_coords = [c for c, v in zip(coords, patch_variances) if v > threshold]
print(
    f"Filtered patches: {len(filtered_coords)} / {len(coords)} (top 40% by variance)")
del raw_data

random.shuffle(filtered_coords)
split = int(len(filtered_coords) * TRAIN_SPLIT)
train_coords = filtered_coords[:split]
val_coords = filtered_coords[split:]
print(f"Train: {len(train_coords)} | Val: {len(val_coords)}")

Loading and multilooing for variance analysis...
Computing patch variances...
Filtered patches: 2778 / 6946 (top 40% by variance)
Train: 2361 | Val: 417


DataSet /. Run this cell before the one over

In [5]:

def multilook(image, looks=4):
    H, W = image.shape
    H_ml = (H // looks) * looks
    W_ml = (W // looks) * looks
    return image[:H_ml, :W_ml].reshape(H // looks, looks, W // looks, looks).mean(axis=(1, 3))


class SARPatchDataset(Dataset):
    def __init__(self, tif_path, coords, patch_size_hr, scale, augment=False):
        self.coords = coords
        self.patch_size_hr = patch_size_hr
        self.scale = scale
        self.augment = augment

        print("Loading TIFF into RAM...")
        with tifffile.TiffFile(tif_path) as tif:
            data = tif.pages[0].asarray()

        if np.iscomplexobj(data):
            data = np.abs(data).astype(np.float32)
        else:
            data = data.astype(np.float32)

        print("Applying multilook...")
        data = multilook(data, looks=LOOKS)
        print(f"After multilook shape: {data.shape}")

        data = np.log1p(data)
        data = (data - data.min()) / (data.max() - data.min() + 1e-8)
        self.image = data
        print(f"Ready — RAM: ~{self.image.nbytes / 1e6:.0f} MB")

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        y, x = self.coords[idx]
        H, W = self.image.shape
        y = min(y, H - self.patch_size_hr)
        x = min(x, W - self.patch_size_hr)

        hr = self.image[y:y+self.patch_size_hr, x:x+self.patch_size_hr].copy()

        if self.augment:
            # Random flips
            if random.random() > 0.5:
                hr = np.fliplr(hr).copy()
            if random.random() > 0.5:
                hr = np.flipud(hr).copy()
            # Random 90 degree rotations
            k = random.randint(0, 3)
            if k > 0:
                hr = np.rot90(hr, k).copy()

        ps_lr = self.patch_size_hr // self.scale
        lr = hr.reshape(ps_lr, self.scale, ps_lr, self.scale).mean(axis=(1, 3))

        lr = torch.from_numpy(lr.astype(np.float32)).unsqueeze(0)
        hr = torch.from_numpy(hr.astype(np.float32)).unsqueeze(0)
        return lr, hr

Ram issue so this fixed it


In [7]:

train_dataset = SARPatchDataset(
    TIF_PATH, train_coords, PATCH_SIZE_HR, SCALE_FACTOR, augment=True)
val_dataset = SARPatchDataset(
    TIF_PATH, val_coords,   PATCH_SIZE_HR, SCALE_FACTOR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

Loading TIFF into RAM...
Applying multilook...
After multilook shape: (3049, 9745)
Ready — RAM: ~119 MB
Loading TIFF into RAM...
Applying multilook...
After multilook shape: (3049, 9745)
Ready — RAM: ~119 MB


In [8]:
STEPS_PER_EPOCH = len(train_dataset) // BATCH_SIZE
print(f"Train patches : {len(train_dataset)}")
print(f"Val patches   : {len(val_dataset)}")
print(f"Steps/epoch   : {STEPS_PER_EPOCH}")
print(f"Total epochs  : {EPOCHS}")
print(f"Total steps   : {STEPS_PER_EPOCH * EPOCHS:,}")

Train patches : 2361
Val patches   : 417
Steps/epoch   : 147
Total epochs  : 500
Total steps   : 73,500


# MODEL

In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, filters=64, res_scale=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(filters, filters, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(filters, filters, 3, padding=1),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.block(x) * self.res_scale

In [10]:
class EDSR(nn.Module):
    def __init__(self, scale=4, num_blocks=16, filters=64):
        super().__init__()
        self.head = nn.Conv2d(1, filters, 3, padding=1)
        self.body = nn.Sequential(*[ResidualBlock(filters)
                                  for _ in range(num_blocks)])
        self.body_end = nn.Conv2d(filters, filters, 3, padding=1)

        # Upsampling: two x2 pixel shuffle steps
        self.upsample = nn.Sequential(
            nn.Conv2d(filters, filters * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.Conv2d(filters, filters * 4, 3, padding=1),
            nn.PixelShuffle(2),
        )
        self.tail = nn.Conv2d(filters, 1, 3, padding=1)

    def forward(self, x):
        x = self.head(x)
        long_skip = x
        x = self.body(x)
        x = self.body_end(x)
        x = x + long_skip
        x = self.upsample(x)
        x = self.tail(x)
        return torch.clamp(x, 0.0, 1.0)


model = EDSR(scale=4, num_blocks=16, filters=64).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 1,515,265


# Metrics


In [ ]:
model = EDSR(scale=4, num_blocks=16, filters=64).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(DEVICE)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)

os.makedirs("comparisons", exist_ok=True)


def find_interesting_patches(dataset, num_candidates=200, num_select=3):
    """Pick high variance patches — more likely to contain real structure."""
    num_candidates = min(num_candidates, len(
        dataset))
    candidates = random.sample(range(len(dataset)), num_candidates)
    variances = []
    for idx in candidates:
        _, hr = dataset[idx]
        variances.append((hr.numpy().var(), idx))
    variances.sort(reverse=True)
    return [idx for _, idx in variances[:num_select]]

Parameters: 1,515,265


# Save Results 

In [12]:
def save_comparison(model, val_dataset, epoch, device, save_dir="MultiStep2", num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

    indices = find_interesting_patches(
        val_dataset, num_candidates=500, num_select=num_samples)

    with torch.no_grad():
        for row, idx in enumerate(indices):
            lr, hr = val_dataset[idx]
            pred = model(lr.unsqueeze(0).to(device)).squeeze().cpu().numpy()
            lr_np = lr.squeeze().numpy()
            hr_np = hr.squeeze().numpy()

            lr_upscaled = np.kron(lr_np, np.ones((SCALE_FACTOR, SCALE_FACTOR)))

            pred_t = torch.from_numpy(pred).unsqueeze(
                0).unsqueeze(0).to(device)
            hr_t = torch.from_numpy(hr_np).unsqueeze(0).unsqueeze(0).to(device)
            patch_psnr = psnr_metric(pred_t, hr_t).item()
            patch_ssim = ssim_metric(pred_t, hr_t).item()

            axes[row, 0].imshow(lr_upscaled, cmap="gray", vmin=0, vmax=1)
            axes[row, 0].set_title(
                f"LR (upscaled) — {lr_np.shape[0]*SCALE_FACTOR}x{lr_np.shape[1]*SCALE_FACTOR}")
            axes[row, 0].axis("off")

            axes[row, 1].imshow(pred, cmap="gray", vmin=0, vmax=1)
            axes[row, 1].set_title(f"EDSR Output\nPSNR: {patch_psnr:.2f} dB | SSIM: {patch_ssim:.4f}",
                                   fontweight="bold")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(hr_np, cmap="gray", vmin=0, vmax=1)
            axes[row, 2].set_title(
                f"HR Ground Truth — {hr_np.shape[0]}x{hr_np.shape[1]}")
            axes[row, 2].axis("off")

    plt.suptitle(f"Epoch {epoch}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(save_dir, f"epoch_{epoch:03d}.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Comparison saved: {path}")

# Training

In [ ]:
model.eval()
lr_batch, hr_batch = next(iter(train_loader))
print("LR batch:", lr_batch.shape)
print("HR batch:", hr_batch.shape)
with torch.no_grad():
    pred = model(lr_batch.to(DEVICE))
print("Pred shape:", pred.shape)
print("Pred range:", pred.min().item(), "–", pred.max().item())


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=5
)
criterion = nn.L1Loss()

best_val_loss = float("inf")
STEPS_PER_EPOCH = len(train_dataset) // BATCH_SIZE
print(f"Steps per epoch: {STEPS_PER_EPOCH}")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_iter = iter(train_loader)

    pbar = tqdm(range(STEPS_PER_EPOCH),
                desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for step in pbar:
        try:
            lr_batch, hr_batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            lr_batch, hr_batch = next(train_iter)

        lr_batch = lr_batch.to(DEVICE)
        hr_batch = hr_batch.to(DEVICE)

        optimizer.zero_grad()
        pred = model(lr_batch)
        loss = criterion(pred, hr_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         avg=f"{train_loss/(step+1):.4f}")

    train_loss /= STEPS_PER_EPOCH

    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0

    with torch.no_grad():
        pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ")
        for lr_batch, hr_batch in pbar_val:
            lr_batch = lr_batch.to(DEVICE)
            hr_batch = hr_batch.to(DEVICE)
            pred = model(lr_batch)

            val_loss += criterion(pred, hr_batch).item()
            val_psnr += psnr_metric(pred, hr_batch).item()
            val_ssim += ssim_metric(pred, hr_batch).item()

            pbar_val.set_postfix(
                loss=f"{val_loss/max(1, pbar_val.n):.4f}",
                psnr=f"{val_psnr/max(1, pbar_val.n):.2f}",
                ssim=f"{val_ssim/max(1, pbar_val.n):.4f}",
            )

    n_val = len(val_loader)
    val_loss /= n_val
    val_psnr /= n_val
    val_ssim /= n_val

    scheduler.step(val_loss)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS} Summary")
    print(f"  Train Loss : {train_loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  Val PSNR   : {val_psnr:.2f} dB")
    print(f"  Val SSIM   : {val_ssim:.4f}")
    print(f"  LR         : {optimizer.param_groups[0]['lr']:.2e}")
    print(f"{'='*60}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "edsr_sar_x4_best.pth")
        print(f"New best model saved (val loss: {val_loss:.4f})\n")

    if (epoch + 1) % 5 == 0:
        save_comparison(model, val_dataset, epoch + 1, DEVICE)

LR batch: torch.Size([16, 1, 32, 32])
HR batch: torch.Size([16, 1, 128, 128])
Pred shape: torch.Size([16, 1, 128, 128])
Pred range: 0.0 – 0.0837864875793457
Steps per epoch: 147


Epoch 1/500 [Val]  : 100%|██████████| 27/27 [00:00<00:00, 57.91it/s, loss=0.0836, psnr=22.50, ssim=0.1105] 



Epoch 1/500 Summary
  Train Loss : 0.0964
  Val Loss   : 0.0712
  Val PSNR   : 19.17 dB
  Val SSIM   : 0.0941
  LR         : 1.00e-04

New best model saved (val loss: 0.0712)



Epoch 2/500 [Val]  : 100%|██████████| 27/27 [00:00<00:00, 108.61it/s, loss=0.0296, psnr=37.07, ssim=0.6824] 



Epoch 2/500 Summary
  Train Loss : 0.0393
  Val Loss   : 0.0241
  Val PSNR   : 30.20 dB
  Val SSIM   : 0.5560
  LR         : 1.00e-04

New best model saved (val loss: 0.0241)



...


Epoch 498/500 Summary
  Train Loss : 0.0229
  Val Loss   : 0.0228
  Val PSNR   : 30.72 dB
  Val SSIM   : 0.5922
  LR         : 1.22e-08

New best model saved (val loss: 0.0228)



Epoch 499/500 [Val]  : 100%|██████████| 27/27 [00:00<00:00, 110.65it/s, loss=0.0280, psnr=37.70, ssim=0.7268] 



Epoch 499/500 Summary
  Train Loss : 0.0229
  Val Loss   : 0.0228
  Val PSNR   : 30.72 dB
  Val SSIM   : 0.5922
  LR         : 1.22e-08



Epoch 500/500 [Val]  : 100%|██████████| 27/27 [00:00<00:00, 100.30it/s, loss=0.0294, psnr=39.50, ssim=0.7614]



Epoch 500/500 Summary
  Train Loss : 0.0229
  Val Loss   : 0.0228
  Val PSNR   : 30.72 dB
  Val SSIM   : 0.5922
  LR         : 1.22e-08

Comparison saved: MultiStep2\epoch_500.png
